# Implémentation complète d'un réseau de neurones (Partie 2)

Dans la première partie, nous avons vu comment on pouvait implémenter une passe avant dans un réseau de neurones en utilisant une approche orientée objet. Dans cette partie on verra comment on peut s'appuyer sur cette structure objet pour implémenter l'algorithme d'apprentissage d'un réseau de neurones: la rétro-propagation du gradient.

L'implémentation de la rétro-propragation du gradient nécessite plusieurs ajustements de notre code. Dans une première étape, nous allons doter tous nos opérateurs d'une méthode permettant de calculer le gradient et d'en retourner sa valeur.

## Configuration du notebook

On se limite au strict minimum: numpy !

In [241]:
import numpy as np

## Implémentation d'une méthode du calcul du gradient

Commençons par les opérateurs ReLU et logistique:

$$
    \nabla_Z \operatorname{ReLU}(Z) = 
    \begin{cases}
        1 & \text{si} \; Z \ge 0 \\
        0 & \text{sinon}
    \end{cases}
$$

et 

$$
    \nabla_Z \sigma(Z) = \sigma(Z) \left( 1 - \sigma(Z) \right)
$$

Il nous faut définir cette nouvelle méthode pour tous les modules, même s'il n'y a pas de fonction par défaut. On en profite pour introduire une méthode qui permet de rétro-propager le gradient. Mais pour rétro-propager le gradient il est nécessaire, au moment de la passe avant de sauvegarder les entrées X !

Implémentons ces changements:

In [267]:
class Module:
    def __init__(self):
        """
        Constructeur
        """
        self.input_cache = None  # ppur sauvegarder les entrées de la passe avant pour la passe arrière
    
    def reset_parameters(self):
        """
        Initialisation aléatoire des paramètres. On considère que par défaut, il n'y en a pas.
        """
        pass

    def forward(self, X):
        """
        Implémentation de la passe avant à partir des entrées.
        
        - X: entrées, de dimensions (n, m) avec n le nombre d'observations et m le nombre de caractéristiques
        """
        self.input_cache = X

    def __call__(self, X):
        """
        Raccourci pour réaliser une passe avant.
        """
        return self.forward(X)

    def backward(self, grad=None):
        """
        Rétro-propage de la gradient à partir du gradient passé en argument.

        - grad: gradient rétro-propagé
        """
        # On précise que cette méthode DOIT être définie pour chacun des modules, il n'y a pas de comportement par défaut
        raise NotImplementedError

Nous aurons aussi besoin des opérations de multiplication et de soustraction de tenseurs en plus de l'opérateur unaire `-`:

In [268]:
class Tensor:
    def __init__(self, array):
        """
        Constructeur.

        - array: valeur du tenseur (array numpy)
        """
        self.data = array

    @property
    def shape(self):
        return self.data.shape

    @property
    def T(self):
        """
        Opération de transposition.
        """
        return Tensor(self.data.T)
    
    def __neg__(self):
        """
        Opération unaire de négation.
        """
        return Tensor(-self.data)
    
    def __add__(self, other):
        """
        Opération d'addition avec un autre tenseur ou une constante.

        - other: autre tenseur ou constante
        """
        if hasattr(other, "data"):
            return Tensor(self.data + other.data)
        else:
            return Tensor(self.data + other)

    def __radd__(self, other):
        """
        Opération d'addition avec un autre tenseur ou une constante.

        - other: autre tenseur ou constante
        """
        return self.__add__(other)

    def __sub__(self, other):
        """
        Opération de soustraction avec un autre tenseur ou une constante.

        - other: autre tenseur ou constante
        """
        if hasattr(other, "data"):
            return Tensor(self.data - other.data)
        else:
            return Tensor(self.data - other)

    def __rsub__(self, other):
        """
        Opération de soustraction avec un autre tenseur ou une constante.

        - other: autre tenseur ou constante
        """
        return - self.__sub__(other)  # attention au piège ici ;)

    def __mul__(self, other):
        """
        Opération de multiplication (membre à membre) avec un autre tenseur ou une constante.

        - other: autre tenseur ou constante
        """
        if hasattr(other, "data"):
            return Tensor(self.data * other.data)
        else:
            return Tensor(self.data * other)

    def __rmul__(self, other):
        """
        Opération d'addition avec un autre tenseur ou une constante.

        - other: autre tenseur ou constante
        """
        return self.__mul__(other)
    
    def __matmul__(self, other):
        """
        Opération de multiplication matricielle.

        - other: autre tenseur
        """
        return Tensor(self.data @ other.data)

    def __str__(self):
        """
        Retourne une représentation textuelle du tenseur.
        """
        return str(self.data)

    def __repr__(self):
        """
        Retourne une représentation textuelle du tenseur.
        """
        return repr(self.data)

Nos nouveaux opérateurs pour ReLU et la fonction logistique:

In [274]:
class ReLU(Module):
    def forward(self, X):
        """
        Implémentation de la passe avant à partir des entrées

        - X: entrées, de dimensions (n, m) avec n le nombre d'observations et m le nombre de caractéristiques

        Retourne un tenseur de dimensions (n, out_features)
        """
        super().forward(X)  # Il nous faut un appel explicite pour sauvegarder les entrées pour la rétro-propagation
        return Tensor(np.where(X.data >= 0., X.data, 0.))

    def backward(self, grad):
        """
        Rétro-propage de la gradient à partir du gradient passé en argument.

        - grad: gradient rétro-propagé
        """
        # On récupère les entrées sauvegardées pour calculer le gradient
        X = self.input_cache
        _grad = Tensor(np.where(X.data >= 0., 1., 0.))
        # On propage
        return grad * _grad


class Logistic(Module):
    def forward(self, X):
        """
        Implémentation de la passe avant à partir des entrées

        - X: entrées, de dimensions (n, m) avec n le nombre d'observations et m le nombre de caractéristiques

        Retourne un tenseur de dimensions (n, out_features)
        """
        super().forward(X)  # Il nous faut un appel explicite pour sauvegarder les entrées pour la rétro-propagation
        return Tensor(1 / (1 + np.exp(-X.data)))

    def backward(self, grad):
        """
        Rétro-propage de la gradient à partir du gradient passé en argument.

        - grad: gradient rétro-propagé
        """
        # On récupère les entrées sauvegardées pour calculer le gradient
        X = self.input_cache
        Z = self.forward(X)
        _grad = Z * (1 - Z)
        # On propage
        return grad * _grad

On peut maintenant s'attaquer à l'opérateur linéaire. Mais c'est plus complexe !

Dans notre arbre de calcul, la couche linéaire est un noeud de l'arbre qui nécessite de propager le gradient de l'erreur dans 3 directions:

- vers la couche précédente (par rapport aux entrées)
- vers les poids (par rapport aux poids)
- vers les biais (par rapport au biais)

Donc notre méthode `backward` ici va propager le gradient vers la couche précédente. Mais les gradients calculés pour les poids et biais seront stockés directement dans les paramètres du modèles. Nous allons donc introduire une nouvelle classe pour les paramètres. Ce sont des tenseurs pour lesquels on a stocké des gradients.

Il nous reste à calculer les gradients liés à l'opération linéaire. La clé est de bien faire attention aux dimensions des tenseurs que l'on attend !

Le résultat de l'opérateur linéaire pour les entrées $X$ est $n \times \text{out}$. Quand on calcule la dérivée par la matrice de poids $W$ qui est $\text{in} \times \text{out}$ le gradient de l'opérateur linéaire est alors de dimension $n \times \text{out} \times \text{in} \times \text{out}$ soit un tenseur 4D. Alors nous ne voulons pas calculer ce tenseur. Ce qui nous interesse c'est le gradient de l'erreur par rapport aux entrées, au poids et au biais. Ce gradient aura pour dimensions respectives $n \times \text{in}$, $\text{in} \times \text{out}$ et $1 \times \text{out}$, c'est à dire les mêmes dimensions que la quantité par rapport à laquelle on dérive. Ceci est vrai car l'erreur est un scalaire.

En notant le gradient de l'erreur que l'on reçoit pour propagation $\delta \in \mathcal{M}_{n,\text{out}}(\mathbb{R})$:

$$
\begin{aligned}
    \nabla_X \mathcal{L} &= \delta W^T  \\
    \nabla_W \mathcal{L} &= X^T \delta \\
    \nabla_b \mathcal{L} &= 1_n^T \delta
\end{aligned}
$$

In [276]:
class Parameter(Tensor):
    def __init__(self, array):
        super().__init__(array)
        self.grad = None

In [277]:
class Linear(Module):
    def __init__(self, in_features, out_features):
        """
        Constructeur.

        Ceci est la fonction qui sera appelée lorsqu'un objet est crée. self est un argument muet qui permet d'utiliser l'objet depuis 
        les différentes méthodes que l'on implémentera.

        - in_features: nombre de caractéristiques d'entrées pour la couche
        - out_features: nombre de caractéristiques de sortie pour la couche (c'est aussi le nombre de neurones)
        """
        # Cela permet de sauvegarder les arguments sur les nombres de caractéristiques directement dans l'objet, dans des attributs
        self.in_features = in_features
        self.out_features = out_features
        # Les paramètres (poids et biais) ne sont pas initialisés par défaut, leur valeur est indéfinie
        self.weight = None
        self.bias = None
        # On force une initialisation des paramètres
        self.reset_parameters()
        # Gestion de l'héritage
        super().__init__()

    def reset_parameters(self):
        """
        Initialisation aléatoire des paramètres (poids et biais).
        """
        self.weight = Parameter(np.random.normal(0., 1., (self.in_features, self.out_features)))
        self.bias = Parameter(np.random.normal(0., 1., (1, self.out_features)))

    def forward(self, X):
        """
        Implémentation de la passe avant à partir des entrées
        
        - X: entrées, de dimensions (n, in_features) avec n le nombre d'observations

        Retourne un tenseur de dimensions (n, out_features)
        """
        super().forward(X)  # Il nous faut un appel explicite pour sauvegarder les entrées pour la rétro-propagation
        return X @ self.weight + self.bias

    def backward(self, grad):
        """
        Rétro-propage de la gradient à partir du gradient passé en argument.

        - grad: gradient rétro-propagé
        """
        # On récupère les entrées de la passe avant
        X = self.input_cache
        n = X.shape[0]

        # On calcule et on stocke les gradients par rapport aux poids et biais
        self.weight.grad = X.T @ grad
        self.bias.grad = Tensor(np.ones((1, n))) @ grad
        
        # On calcule le gradient des erreurs par rapport aux entrées et on le propage
        grad_X = grad @ self.weight.T
        return grad_X

Enfin on rajoute la méthode backward à la séquence:

In [278]:
class Sequential(Module):
    def __init__(self, modules):
        """
        Constructeur.

        - modules: liste des modules à exécuter en séquence
        """
        self.modules = modules
        super().__init__()

    def reset_parameters(self):
        """
        Initialisation de tous les paramètres des modules de la séquence.
        """
        for module in self.modules:
            module.reset_parameters()

    def forward(self, X):
        """
        Implémentation de la passe avant à partir des entrées.
        
        - X: entrées, de dimensions (n, m) avec n le nombre d'observations et m le nombre de caractéristiques
        """
        for module in self.modules:
            X = module.forward(X)
        return X

    def backward(self, grad):
        """
        Rétro-propage de la gradient à partir du gradient passé en argument.

        - grad: gradient rétro-propagé
        """
        for module in self.modules[-1::-1]:
            grad = module.backward(grad)
        return grad

Pour illustrer ce qu'il se passe, prennons un algorithme de classification:

In [279]:
n = 100
m0 = 2
X0 = Tensor(np.random.normal(0., 1., (n, m0)))

In [280]:
linear1 = Linear(2, 8)
linear2 = Linear(8, 1)
relu = ReLU()
logistic = Logistic()
model = Sequential([linear1, relu, linear2, logistic])

Supposons que le gradient des erreurs que l'on calcule sur la dernière couche est delta avec comme dimension $n \times 1$ puisque c'est le nombre d'observations fois le nombre de caractéristiques de sortie pour notre modèle.

In [281]:
delta = Tensor(np.random.normal(0., 1., (n, 1)))

On commence par une passe avant:

In [282]:
Y = model(X0)

Et maintenant on retro-propage le gradient des erreurs $\delta$:

In [283]:
grad = model.backward(delta)
grad.shape

(100, 2)

Ce dernier tenseur retourné contient la dérivée des erreurs en fonction des caractéristiques d'entrées ! Il a donc la même dimension que la matrice des observations $X_0$.

## Extraction des paramètres

Ok, nous avons maintenant une vraie machine de calcul ! Il nous reste un dernier élément dont nous allons avoir besoin pour implémenter l'algorithme d'apprentissage, il ous faut être capable de récupérer les gradients stockés au niveau des paramètres.

On a donc besoin d'un ajustement des classes `Module` et `Sequential`:

In [284]:
class Module:
    def __init__(self):
        """
        Constructeur
        """
        self.input_cache = None  # pour sauvegarder les entrées de la passe avant pour la passe arrière
    
    def reset_parameters(self):
        """
        Initialisation aléatoire des paramètres. On considère que par défaut, il n'y en a pas.
        """
        pass

    def forward(self, X):
        """
        Implémentation de la passe avant à partir des entrées.
        
        - X: entrées, de dimensions (n, m) avec n le nombre d'observations et m le nombre de caractéristiques
        """
        self.input_cache = X

    def __call__(self, X):
        """
        Raccourci pour réaliser une passe avant.
        """
        return self.forward(X)

    def backward(self, grad):
        """
        Rétro-propage de la gradient à partir du gradient passé en argument.

        - grad: gradient rétro-propagé
        """
        # On précise que cette méthode DOIT être définie pour chacun des modules, il n'y a pas de comportement par défaut
        raise NotImplementedError

    def parameters(self):
        """
        Retourne une liste contenant tous les paramètres 'apprenables' du module.
        """
        params = []
        # On parcourt tous les attributs de l'objet
        for key, value in self.__dict__.items():
            # Si l'attribut est un Paramètre, on l'ajoute à la liste
            if isinstance(value, Parameter):
                params.append(value)
            # Si l'attribut est lui-même un sous-module, on va chercher ses paramètres de façon récursive
            elif isinstance(value, Module):
                params.extend(value.parameters())
        return params

In [285]:
class Sequential(Module):
    def __init__(self, modules):
        """
        Constructeur.

        - modules: liste des modules à exécuter en séquence
        """
        self.modules = modules
        super().__init__()

    def reset_parameters(self):
        """
        Initialisation de tous les paramètres des modules de la séquence.
        """
        for module in self.modules:
            module.reset_parameters()

    def forward(self, X):
        """
        Implémentation de la passe avant à partir des entrées.
        
        - X: entrées, de dimensions (n, m) avec n le nombre d'observations et m le nombre de caractéristiques
        """
        for module in self.modules:
            X = module.forward(X)
        return X

    def backward(self, grad):
        """
        Rétro-propage de la gradient à partir du gradient passé en argument.

        - grad: gradient rétro-propagé
        """
        for module in self.modules[-1::-1]:
            grad = module.backward(grad)
        return grad

    def parameters(self):
        """
        Retourne une liste contenant tous les paramètres apprenables de la séquence.
        """
        params = []
        for module in self.modules:
            params.extend(module.parameters())
        return params   

Il nous faut malheureusement re-exécuter le code sur les modules vue que l'on a changé la classe de base:

In [286]:
class ReLU(Module):
    def forward(self, X):
        """
        Implémentation de la passe avant à partir des entrées

        - X: entrées, de dimensions (n, m) avec n le nombre d'observations et m le nombre de caractéristiques

        Retourne un tenseur de dimensions (n, out_features)
        """
        super().forward(X)  # Il nous faut un appel explicite pour sauvegarder les entrées pour la rétro-propagation
        return Tensor(np.where(X.data >= 0., X.data, 0.))

    def backward(self, grad):
        """
        Rétro-propage de la gradient à partir du gradient passé en argument.

        - grad: gradient rétro-propagé
        """
        # On récupère les entrées sauvegardées pour calculer le gradient
        X = self.input_cache
        _grad = Tensor(np.where(X.data >= 0., 1., 0.))
        # On propage
        return grad * _grad


class Logistic(Module):
    def forward(self, X):
        """
        Implémentation de la passe avant à partir des entrées

        - X: entrées, de dimensions (n, m) avec n le nombre d'observations et m le nombre de caractéristiques

        Retourne un tenseur de dimensions (n, out_features)
        """
        super().forward(X)  # Il nous faut un appel explicite pour sauvegarder les entrées pour la rétro-propagation
        return Tensor(1 / (1 + np.exp(-X.data)))

    def backward(self, grad):
        """
        Rétro-propage de la gradient à partir du gradient passé en argument.

        - grad: gradient rétro-propagé
        """
        # On récupère les entrées sauvegardées pour calculer le gradient
        X = self.input_cache
        Z = self.forward(X)
        _grad = Z * (1 - Z)
        # On propage
        return grad * _grad


class Linear(Module):
    def __init__(self, in_features, out_features):
        """
        Constructeur.

        Ceci est la fonction qui sera appelée lorsqu'un objet est crée. self est un argument muet qui permet d'utiliser l'objet depuis 
        les différentes méthodes que l'on implémentera.

        - in_features: nombre de caractéristiques d'entrées pour la couche
        - out_features: nombre de caractéristiques de sortie pour la couche (c'est aussi le nombre de neurones)
        """
        # Cela permet de sauvegarder les arguments sur les nombres de caractéristiques directement dans l'objet, dans des attributs
        self.in_features = in_features
        self.out_features = out_features
        # Les paramètres (poids et biais) ne sont pas initialisés par défaut, leur valeur est indéfinie
        self.weight = None
        self.bias = None
        # On force une initialisation des paramètres
        self.reset_parameters()
        # Gestion de l'héritage
        super().__init__()

    def reset_parameters(self):
        """
        Initialisation aléatoire des paramètres (poids et biais).
        """
        self.weight = Parameter(np.random.normal(0., 1., (self.in_features, self.out_features)))
        self.bias = Parameter(np.random.normal(0., 1., (1, self.out_features)))

    def forward(self, X):
        """
        Implémentation de la passe avant à partir des entrées
        
        - X: entrées, de dimensions (n, in_features) avec n le nombre d'observations

        Retourne un tenseur de dimensions (n, out_features)
        """
        super().forward(X)  # Il nous faut un appel explicite pour sauvegarder les entrées pour la rétro-propagation
        return X @ self.weight + self.bias

    def backward(self, grad):
        """
        Rétro-propage de la gradient à partir du gradient passé en argument.

        - grad: gradient rétro-propagé
        """
        # On récupère les entrées de la passe avant
        X = self.input_cache
        n = X.shape[0]

        # On calcule et on stocke les gradients par rapport aux poids et biais
        self.weight.grad = X.T @ grad
        self.bias.grad = Tensor(np.ones((1, n))) @ grad
        
        # On calcule le gradient des erreurs par rapport aux entrées et on le propage
        grad_X = grad @ self.weight.T
        return grad_X

Application à notre cas de classification:

In [259]:
n = 100
m0 = 2
X0 = Tensor(np.random.normal(0., 1., (n, m0)))

In [260]:
linear1 = Linear(2, 8)
linear2 = Linear(8, 1)
relu = ReLU()
logistic = Logistic()
model = Sequential([linear1, relu, linear2, logistic])

In [261]:
delta = Tensor(np.random.normal(0., 1., (n, 1)))

In [262]:
Y = model(X0)

In [263]:
grad = model.backward(delta)

On peut récupérer les paramètres qui nous interessent, ce que l'on veut optimiser !

In [264]:
parameters = model.parameters()
parameters

[array([[ 0.48468617, -0.43274485,  0.09495991,  0.4127549 , -3.02774606,
          0.56479956,  2.15806926,  0.07587922],
        [-0.17597251, -0.56311544,  0.6037599 ,  0.39416618,  1.67713493,
          0.49806806, -0.92633022, -0.88448269]]),
 array([[ 0.41203736,  0.09661554, -0.02286925, -0.21561383, -0.85266945,
          0.10739214, -0.26738503,  1.44600378]]),
 array([[ 1.0851926 ],
        [ 0.63152137],
        [-1.16607756],
        [ 0.49697315],
        [-0.09765182],
        [ 3.56776284],
        [ 1.71344433],
        [ 1.05945862]]),
 array([[0.84230757]])]

Nous avons récupéré les 4 paramètres: les poids et biais des deux couches !

Nous pouvons regarder les gradients qui ont été accumulés dans ces paramètres:

In [265]:
for param in parameters:
    print(param.grad)

[[-0.07750255 -0.38760381  1.52783681 -0.11538585  0.1255233  -2.23155401
   0.00644347 -1.01939071]
 [-0.07635431  0.28307786 -1.72667089  0.24394017 -0.14140509  3.71098916
   0.04667079  0.84980351]]
[[-0.0513805   0.36510547 -1.46870824  0.07679755 -0.13027921  2.43129794
  -0.02204567  1.00208027]]
[[-0.04174267]
 [ 0.06904422]
 [ 0.74079391]
 [ 0.06432596]
 [ 5.18293429]
 [ 0.23797735]
 [-0.01367562]
 [ 0.58522795]]
[[1.2942594]]


Nous sommes prêts pour passer à l'optimisation !